# Notebook 06: Active Learning

**Module:** 04 ML Paradigms  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Define active learning
2. Implement uncertainty sampling
3. Simulate labeling budget constraints
4. Apply to GeoSpatial annotation

## 1. Intuition

**Problem:** Labeling 10,000 satellite tiles costs $50,000. You have budget for 500 labels.

**Active learning:** Model **chooses** which samples to label next — picks most informative ones.

**Query strategies:**
- **Uncertainty sampling:** label samples where model is least confident
- **Margin sampling:** smallest difference between top-2 class probabilities
- **Query-by-committee:** models disagree most

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path('../../').resolve()
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(42)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_pool, X_test, y_pool, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Start with 10 labeled samples
labeled_idx = rng.choice(len(X_pool), 10, replace=False)
unlabeled_mask = np.ones(len(X_pool), dtype=bool)
unlabeled_mask[labeled_idx] = False

accuracies = []
for iteration in range(20):
    X_l = X_pool[labeled_idx]
    y_l = y_pool[labeled_idx]
    clf = LogisticRegression(max_iter=1000).fit(X_l, y_l)
    acc = clf.score(X_test, y_test)
    accuracies.append(acc)
    
    # Uncertainty sampling: pick least confident unlabeled sample
    X_unlabeled = X_pool[unlabeled_mask]
    if len(X_unlabeled) == 0: break
    proba = clf.predict_proba(X_unlabeled)
    uncertainty = 1 - proba.max(axis=1)
    pick = uncertainty.argmax()
    unlabeled_indices = np.where(unlabeled_mask)[0]
    labeled_idx = np.append(labeled_idx, unlabeled_indices[pick])
    unlabeled_mask[unlabeled_indices[pick]] = False

plt.plot(range(10, 10+len(accuracies)), accuracies, 'o-')
plt.xlabel('Number of labeled samples'); plt.ylabel('Test accuracy')
plt.title('Active Learning — Uncertainty Sampling'); plt.show()

## GeoSpatial: Label the 500 most uncertain pond tiles instead of random 500 → better model with same budget.

---

## Interview Questions

See module quiz.

## Summary

Active learning maximizes model quality per labeling dollar.

**Next:** [07_Online_Learning.ipynb](07_Online_Learning.ipynb)